<a href="https://colab.research.google.com/github/dorcasojo/Machine-learning-Assignment/blob/main/Water_Prediction_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#library

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold, TimeSeriesSplit, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score

**Load Dataset**

In [2]:
df = pd.read_csv("/content/Aquifer_Petrignano.csv")
df.head()

,Date,Rainfall_Bastia_Umbra,Depth_to_Groundwater_P24,Depth_to_Groundwater_P25,Temperature_Bastia_Umbra,Temperature_Petrignano,Volume_C10_Petrignano,Hydrometry_Fiume_Chiascio_Petrignano
0,14/03/2006,NaN,-22.48,-22.18,NaN,NaN,NaN,NaN
1,15/03/2006,NaN,-22.38,-22.14,NaN,NaN,NaN,NaN
2,16/03/2006,NaN,-22.25,-22.04,NaN,NaN,NaN,NaN
3,17/03/2006,NaN,-22.38,-22.04,NaN,NaN,NaN,NaN
4,18/03/2006,NaN,-22.60,-22.04,NaN,NaN,NaN,NaN


In [3]:
# Convert Date column to datetime
df['Date'] = pd.to_datetime(df['Date'])

/tmp/ipykernel_3871/2690484855.py:2: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['Date'] = pd.to_datetime(df['Date'])


***PROMPT: Ensure chronlogical order and split into traning and test***

In [4]:
# Sort chronologically
df = df.sort_values('Date')
df.head()

,Date,Rainfall_Bastia_Umbra,Depth_to_Groundwater_P24,Depth_to_Groundwater_P25,Temperature_Bastia_Umbra,Temperature_Petrignano,Volume_C10_Petrignano,Hydrometry_Fiume_Chiascio_Petrignano
0,2006-03-14,NaN,-22.48,-22.18,NaN,NaN,NaN,NaN
1,2006-03-15,NaN,-22.38,-22.14,NaN,NaN,NaN,NaN
2,2006-03-16,NaN,-22.25,-22.04,NaN,NaN,NaN,NaN
3,2006-03-17,NaN,-22.38,-22.04,NaN,NaN,NaN,NaN
4,2006-03-18,NaN,-22.60,-22.04,NaN,NaN,NaN,NaN


In [5]:
# Set Date as index
df.set_index('Date', inplace=True)

***PROMPT; How can i handle missing value***

In [6]:
# Replace known problematic zeros with NaN
df.replace(0, np.nan, inplace=True)
df.head()

,Rainfall_Bastia_Umbra,Depth_to_Groundwater_P24,Depth_to_Groundwater_P25,Temperature_Bastia_Umbra,Temperature_Petrignano,Volume_C10_Petrignano,Hydrometry_Fiume_Chiascio_Petrignano
Date,,,,,,,
2006-03-14,NaN,-22.48,-22.18,NaN,NaN,NaN,NaN
2006-03-15,NaN,-22.38,-22.14,NaN,NaN,NaN,NaN
2006-03-16,NaN,-22.25,-22.04,NaN,NaN,NaN,NaN
2006-03-17,NaN,-22.38,-22.04,NaN,NaN,NaN,NaN
2006-03-18,NaN,-22.60,-22.04,NaN,NaN,NaN,NaN


PROMPT; How to fill missing values in time series

Modification; I use method='time' for interpolation instead of linear and did backfill and forward fill

In [9]:
# Interpolate missing values
df = df.interpolate(method='time')

df = df.fillna(method='bfill').fillna(method='ffill')
df.head()

df = df.fillna(method='bfill').fillna(method='ffill')
df.head()

/tmp/ipykernel_3871/1297298650.py:4: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method='bfill').fillna(method='ffill')
/tmp/ipykernel_3871/1297298650.py:7: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method='bfill').fillna(method='ffill')


,Rainfall_Bastia_Umbra,Depth_to_Groundwater_P24,Depth_to_Groundwater_P25,Temperature_Bastia_Umbra,Temperature_Petrignano,Volume_C10_Petrignano,Hydrometry_Fiume_Chiascio_Petrignano
Date,,,,,,,
2006-03-14,0.9,-22.48,-22.18,5.2,4.9,-29281.824,2.4
2006-03-15,0.9,-22.38,-22.14,5.2,4.9,-29281.824,2.4
2006-03-16,0.9,-22.25,-22.04,5.2,4.9,-29281.824,2.4
2006-03-17,0.9,-22.38,-22.04,5.2,4.9,-29281.824,2.4
2006-03-18,0.9,-22.60,-22.04,5.2,4.9,-29281.824,2.4


PROMPT; **I have a daily groundwater data , i want to predict next months depth using two previous month of all varaiables, how do i reshape my data frame give example.**

target = 'Depth_to_Groundwater_P25'

In [11]:
# Create lag features for all columns
lag_1 = df.shift(1).add_suffix('_lag1')
lag_2 = df.shift(2).add_suffix('_lag2')
df_lagged = pd.concat([lag_1, lag_2], axis=1)

In [18]:
# Add target column (current DP25)
target = 'Depth_to_Groundwater_P25'
df_lagged['target'] = df[target]

# Drop rows with NaN due to lagging
df_lagged = df_lagged.dropna()

df_lagged.head()

,Rainfall_Bastia_Umbra_lag1,Depth_to_Groundwater_P24_lag1,Depth_to_Groundwater_P25_lag1,Temperature_Bastia_Umbra_lag1,Temperature_Petrignano_lag1,Volume_C10_Petrignano_lag1,Hydrometry_Fiume_Chiascio_Petrignano_lag1,Rainfall_Bastia_Umbra_lag2,Depth_to_Groundwater_P24_lag2,Depth_to_Groundwater_P25_lag2,Temperature_Bastia_Umbra_lag2,Temperature_Petrignano_lag2,Volume_C10_Petrignano_lag2,Hydrometry_Fiume_Chiascio_Petrignano_lag2,target
Date,,,,,,,,,,,,,,,
2006-03-16,0.9,-22.38,-22.14,5.2,4.9,-29281.824,2.4,0.9,-22.48,-22.18,5.2,4.9,-29281.824,2.4,-22.04
2006-03-17,0.9,-22.25,-22.04,5.2,4.9,-29281.824,2.4,0.9,-22.38,-22.14,5.2,4.9,-29281.824,2.4,-22.04
2006-03-18,0.9,-22.38,-22.04,5.2,4.9,-29281.824,2.4,0.9,-22.25,-22.04,5.2,4.9,-29281.824,2.4,-22.04
2006-03-19,0.9,-22.60,-22.04,5.2,4.9,-29281.824,2.4,0.9,-22.38,-22.04,5.2,4.9,-29281.824,2.4,-21.95
2006-03-20,0.9,-22.35,-21.95,5.2,4.9,-29281.824,2.4,0.9,-22.60,-22.04,5.2,4.9,-29281.824,2.4,-21.99


Define predictors (X) and response variable (y)

In [19]:
X = df_lagged.drop(columns=['target'])
y = df_lagged['target']

PROMPT;Split data set into traninig and test without shuffling

In [20]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

*PROMPT;Set up kfold and time series split 5 folds each

*MODIFICATION; I kept random state=42 for reproducibility in the shuffled version

In [21]:
cv_naive = KFold(n_splits=5, shuffle=True, random_state=42)
cv_temporal = TimeSeriesSplit(n_splits=5)

***PROMPT;Generate Python code to create a regression pipeline with scaling and a decision tree model, then evaluate it using both shuffled KFold and TimeSeriesSplit cross-validation for a time series dataset***

In [22]:
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', DecisionTreeRegressor(max_depth=10, random_state=42))
])
pipe

Pipeline(steps=[('scaler', StandardScaler()),
                ('regressor',
                 DecisionTreeRegressor(max_depth=10, random_state=42))])

**PROMPT;** **EVALUATE THE PIPELINE USING NAIVE KFOLD AND TIMESERIESSPLIT CROSS VALIDATION ON THE TRANING DATA AND TRAIN ON FULL TRANING AND EVALUATE THE TEST**

In [23]:
scores_naive = cross_val_score(pipe, X_train_full, y_train_full, cv=cv_naive, scoring='r2')
scores_temporal = cross_val_score(pipe, X_train_full, y_train_full, cv=cv_temporal, scoring='r2')

print(f"Naive CV R2:    {scores_naive.mean():.4f} (+/- {scores_naive.std():.4f})")
print(f"Temporal CV R2: {scores_temporal.mean():.4f} (+/- {scores_temporal.std():.4f})")

# Train final model
pipe.fit(X_train_full, y_train_full)
final_test_r2 = r2_score(y_test, pipe.predict(X_test))

print(f"\nActual Test R2: {final_test_r2:.4f}")

Naive CV R2:    0.9995 (+/- 0.0000)
Temporal CV R2: 0.7206 (+/- 0.4184)

Actual Test R2: 0.9877


**PROMPT;Function for model selection with grid search on tree depth*

**MODIFICATION; I addedd n_job=-1 to speed up and change scoring to 'r2'specifically

In [24]:
def evaluate_model_selection(X_train, y_train, X_test, y_test, cv_strategy, name):


    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', DecisionTreeRegressor(random_state=42))
    ])


    param_grid = {
        'regressor__max_depth': [3, 5, 10, 15]
    }


    grid = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        cv=cv_strategy,
        scoring='r2',
        n_jobs=-1
    )

    grid.fit(X_train, y_train)


    y_pred = grid.predict(X_test)
    test_r2 = r2_score(y_test, y_pred)

    print(f"\n===== Results for: {name} =====")
    print(f"Best Parameters: {grid.best_params_}")
    print(f"CV Score (R2): {grid.best_score_:.4f}")
    print(f"Test Score (R2): {test_r2:.4f}")

    return grid

*PROMPT;Run grid search for both cv methods and compare results

MODIFICATION; I changed the parameters grid from[3,5,10,15] to include max_depth=15 after seeing temporal CV preferred deeper treesbold text*

In [25]:
result_naive = evaluate_model_selection(
    X_train_full, y_train_full, X_test, y_test,
    cv_naive, "Naive K-Fold"
)

result_temporal = evaluate_model_selection(
    X_train_full, y_train_full, X_test, y_test,
    cv_temporal, "TimeSeriesSplit"
)


===== Results for: Naive K-Fold =====
Best Parameters: {'regressor__max_depth': 10}
CV Score (R2): 0.9995
Test Score (R2): 0.9877

===== Results for: TimeSeriesSplit =====
Best Parameters: {'regressor__max_depth': 15}
CV Score (R2): 0.7209
Test Score (R2): 0.9869
